# Closures

## Time Elapsed Since Creating a Closure

Write a function `make_timer()` that returns a closure: each time it is called, the closure returns the time elapsed, in seconds, since it was created. Measure this time with `time.perf_counter`, which is made for durations: unlike `time.time`, it doesn't follow the adjustments of the system clock.

Create `timer = make_timer()`, wait with `sleep(1)`, print `timer()`, then do it again: you should get about 1, then about 2.

**Useful documentation**

- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)
- [`time.sleep`](https://docs.python.org/3/library/time.html#time.sleep)

In [ ]:
# Your code here

### Additional hints

In `make_timer`, keep `perf_counter()` in a variable `start`, then define and return an inner function that computes the difference with a new call of `perf_counter()`. It only reads `start`: there is nothing to declare.

```python
from collections.abc import Callable
from time import perf_counter, sleep


def make_timer() -> Callable[[], float]:
  start = perf_counter()

  def timer() -> float:
    return ...

  return timer
```

### Solution

In [ ]:
from collections.abc import Callable
from time import perf_counter, sleep


def make_timer() -> Callable[[], float]:
  start = perf_counter()

  def timer() -> float:
    return perf_counter() - start

  return timer


timer = make_timer()
sleep(1)
print(timer())
sleep(1)
print(timer())

You can get the names of the variables the closure captures in `__code__.co_freevars`, and their values in `__closure__[index].cell_contents`.

In [ ]:
print(timer.__code__.co_freevars, timer.__closure__[0].cell_contents)

## One Counter per Closure

Write a function `make_counter(step)` that returns a closure: each time it is called, the closure adds `step` to its counter, which starts at 0, and returns the new value.

Create `ones = make_counter(1)` and `tens = make_counter(10)`, and check that the two counters evolve separately: `ones()`, `ones()`, `tens()` and `ones()`, called in this order, should return 1, 2, 10 and 3.

In [ ]:
# Your code here

### Additional hints

The closure reassigns the counter: without a `nonlocal` declaration, `value += step` raises an `UnboundLocalError`. Each call of `make_counter` creates its own variables, hence two independent counters.

```python
from collections.abc import Callable


def make_counter(step: int) -> Callable[[], int]:
  value = 0

  def counter() -> int:
    ...
    return value

  return counter
```

### Solution

In [ ]:
from collections.abc import Callable


def make_counter(step: int) -> Callable[[], int]:
  value = 0

  def counter() -> int:
    nonlocal value
    value += step
    return value

  return counter


ones = make_counter(1)
tens = make_counter(10)
print(ones(), ones(), tens(), ones())

## Fixing the Argument of a Function with a Closure

By default, the `int` function interprets the strings passed to it as base 10 numbers. Write a function `make_int_base(base)` that returns a closure: the closure converts a string to an integer in this base. For example, with `int_base_2 = make_int_base(2)`, `int_base_2("101")` is 5.

**Useful documentation**

- [`int`](https://docs.python.org/3/library/functions.html#int)

In [ ]:
# Your code here

### Additional hints

`int` takes the base through its `base` argument: `int("101", base=2)` is 5. The closure only reads `base`, the parameter of the enclosing function.

```python
from collections.abc import Callable


def make_int_base(base: int) -> Callable[[str], int]:
  def int_base(x: str) -> int:
    return ...

  return int_base
```

### Solution

In [ ]:
from collections.abc import Callable


def make_int_base(base: int) -> Callable[[str], int]:
  def int_base(x: str) -> int:
    return int(x, base=base)

  return int_base


int_base_2 = make_int_base(2)
print(int_base_2("101"))
print(int_base_2.__code__.co_freevars, int_base_2.__closure__[0].cell_contents)

To merely fix an argument, the standard library provides [`functools.partial`](https://docs.python.org/3/library/functools.html#functools.partial), with no closure to write:

In [ ]:
from functools import partial

int_base_2 = partial(int, base=2)
print(int_base_2("101"))

## Creating Multiple Closures in a Loop

The `multiplications` list of the next cell should hold the functions that multiply by 0, 1, 2, 3 and 4, and the cell should print `0 2 4 6 8`. Yet it prints `8 8 8 8 8`.

Explain this result in a comment, then fix the list in two ways to get `0 2 4 6 8`: with a default argument, then with a function that makes each closure.

In [ ]:
multiplications = [lambda x: x * i for i in range(5)]
print(*[m(2) for m in multiplications])

In [ ]:
# Your code here

### Additional hints

A fix must give each function its own value of `i`:

- a default argument is evaluated when the function is created;
- each call of a function that takes `i` and returns the `lambda` creates its own variable `i`.

```python
from collections.abc import Callable

multiplications = [lambda x, i=i: ... for i in range(5)]


def make_multiplication(i: int) -> Callable[[int], int]:
  return lambda x: ...


multiplications = [... for i in range(5)]
```

### Solution

The functions capture the variable `i`, not its value: they read it when they are called, when the loop is over and `i` is 4. This is called late binding. It is very important to be aware of this when creating closures.

In [ ]:
multiplications = [lambda x, i=i: x * i for i in range(5)]
print(*[m(2) for m in multiplications])

In [ ]:
from collections.abc import Callable


def make_multiplication(i: int) -> Callable[[int], int]:
  return lambda x: x * i


multiplications = [make_multiplication(i) for i in range(5)]
print(*[m(2) for m in multiplications])